In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/llm-classification-finetuning/sample_submission.csv
/kaggle/input/competitions/llm-classification-finetuning/train.csv
/kaggle/input/competitions/llm-classification-finetuning/test.csv


In [2]:
from __future__ import annotations

import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import MaxAbsScaler, OneHotEncoder

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.max_columns", 200)

DATA_DIR = Path("/kaggle/input/competitions/llm-classification-finetuning/")
RANDOM_STATE = 42
TARGET_NAMES = np.array(["model_a", "model_b", "tie"])

train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")


def parse_chat(value: str) -> str:
    if pd.isna(value):
        return ""
    try:
        turns = json.loads(value)
    except (TypeError, json.JSONDecodeError):
        return str(value)
    if isinstance(turns, list):
        return " <TURN> ".join(str(turn) for turn in turns)
    return str(turns)


def safe_turn_count(value: str) -> int:
    if pd.isna(value):
        return 0
    try:
        turns = json.loads(value)
    except (TypeError, json.JSONDecodeError):
        return 1
    return len(turns) if isinstance(turns, list) else 1


def _unique_ratio(text: str) -> float:
    words = str(text).split()
    return len(set(words)) / max(1, len(words))


def prepare_frame(frame: pd.DataFrame) -> pd.DataFrame:
    prepared = frame.copy()
    for column in ["prompt", "response_a", "response_b"]:
        prepared[f"{column}_text"] = prepared[column].map(parse_chat)
        prepared[f"{column}_chars"] = prepared[f"{column}_text"].str.len()
        prepared[f"{column}_words"] = prepared[f"{column}_text"].str.split().str.len()
        prepared[f"{column}_turns"] = prepared[column].map(safe_turn_count)

    prepared["pair_a_text"] = prepared["prompt_text"] + " <SEP> " + prepared["response_a_text"]
    prepared["pair_b_text"] = prepared["prompt_text"] + " <SEP> " + prepared["response_b_text"]
    prepared["response_chars_diff"] = prepared["response_a_chars"] - prepared["response_b_chars"]
    prepared["response_words_diff"] = prepared["response_a_words"] - prepared["response_b_words"]
    prepared["response_turns_diff"] = prepared["response_a_turns"] - prepared["response_b_turns"]

    # Log-ratio features fight verbosity bias
    prepared["log_resp_chars_ratio"] = (
        np.log1p(prepared["response_a_chars"]) - np.log1p(prepared["response_b_chars"])
    )
    prepared["log_resp_words_ratio"] = (
        np.log1p(prepared["response_a_words"]) - np.log1p(prepared["response_b_words"])
    )
    # Vocabulary richness: unique word ratio per response
    prepared["resp_a_unique_ratio"] = prepared["response_a_text"].map(_unique_ratio)
    prepared["resp_b_unique_ratio"] = prepared["response_b_text"].map(_unique_ratio)
    prepared["unique_ratio_diff"] = prepared["resp_a_unique_ratio"] - prepared["resp_b_unique_ratio"]

    prepared["model_a"] = prepared.get("model_a", "unknown_model_a")
    prepared["model_b"] = prepared.get("model_b", "unknown_model_b")
    prepared["same_model_family"] = prepared["model_a"].eq(prepared["model_b"])
    return prepared


def add_target(frame: pd.DataFrame) -> pd.DataFrame:
    prepared = frame.copy()
    prepared["target"] = np.select(
        [prepared["winner_model_a"].eq(1), prepared["winner_model_b"].eq(1)],
        ["model_a", "model_b"],
        default="tie",
    )
    return prepared


train_prepared = add_target(prepare_frame(train_df))
test_prepared = prepare_frame(test_df)

NUMERIC_FEATURES = [
    "prompt_chars",
    "prompt_words",
    "prompt_turns",
    "response_a_chars",
    "response_a_words",
    "response_a_turns",
    "response_b_chars",
    "response_b_words",
    "response_b_turns",
    "response_chars_diff",
    "response_words_diff",
    "response_turns_diff",
    "log_resp_chars_ratio",
    "log_resp_words_ratio",
    "resp_a_unique_ratio",
    "resp_b_unique_ratio",
    "unique_ratio_diff",
]
CATEGORICAL_FEATURES = ["model_a", "model_b", "same_model_family"]


In [3]:
dataset_summary = pd.DataFrame(
    {
        "rows": [len(train_prepared), len(test_prepared)],
        "unique_model_a": [train_prepared["model_a"].nunique(), test_prepared["model_a"].nunique()],
        "unique_model_b": [train_prepared["model_b"].nunique(), test_prepared["model_b"].nunique()],
        "avg_prompt_words": [train_prepared["prompt_words"].mean(), test_prepared["prompt_words"].mean()],
        "avg_response_a_words": [train_prepared["response_a_words"].mean(), test_prepared["response_a_words"].mean()],
        "avg_response_b_words": [train_prepared["response_b_words"].mean(), test_prepared["response_b_words"].mean()],
    },
    index=["train", "test"],
)

label_distribution = train_prepared["target"].value_counts(normalize=True).rename("share")
model_pair_frequency = (
    train_prepared.groupby(["model_a", "model_b"]).size().sort_values(ascending=False).head(15).rename("match_count")
)
length_bias = (
    train_prepared.groupby("target")[["response_a_words", "response_b_words", "response_words_diff"]]
    .mean()
    .sort_index()
)

print("Dataset summary")
display(dataset_summary.round(2))
print("\nLabel distribution")
display(label_distribution.to_frame().round(4))
print("\nTop model pair frequencies")
display(model_pair_frequency.to_frame())
print("\nAverage length statistics by winning class")
display(length_bias.round(2))

Dataset summary


,rows,unique_model_a,unique_model_b,avg_prompt_words,avg_response_a_words,avg_response_b_words
train,57477,64,64,56.47,212.58,213.42
test,3,1,1,45.67,284.33,192.00



Label distribution


,share
target,
model_a,0.3491
model_b,0.3419
tie,0.3090



Top model pair frequencies


match_count
model_a            model_b                        
claude-2.1         gpt-4-1106-preview          557
gpt-4-1106-preview claude-2.1                  516
                   gpt-4-0613                  502
gpt-4-0613         gpt-4-1106-preview          480
claude-2.1         claude-1                    412
                   gpt-4-0613                  392
gpt-3.5-turbo-0613 gpt-4-1106-preview          370
gpt-4-0613         claude-2.1                  366
claude-1           claude-2.1                  351
gpt-4-1106-preview gpt-3.5-turbo-0613          337
gpt-4-0613         gpt-4-0314                  290
gpt-3.5-turbo-1106 gpt-4-1106-preview          289
gpt-4-1106-preview gpt-3.5-turbo-1106          287
gpt-4-0314         gpt-4-0613                  270
                   claude-2.1                  250


Average length statistics by winning class


,response_a_words,response_b_words,response_words_diff
target,,,
model_a,241.01,202.37,38.64
model_b,199.30,240.73,-41.43
tie,195.15,195.68,-0.53


In [4]:
def top_chi2_terms(texts: pd.Series, labels: pd.Series, *, k: int = 15, min_df: int = 20) -> pd.DataFrame:
    vectorizer = TfidfVectorizer(
        min_df=min_df,
        max_df=0.97,
        ngram_range=(1, 2),
        sublinear_tf=True,
        strip_accents="unicode",
    )
    matrix = vectorizer.fit_transform(texts)
    feature_names = np.asarray(vectorizer.get_feature_names_out())
    rows: list[dict[str, object]] = []

    for label in TARGET_NAMES:
        scores, _ = chi2(matrix, labels.eq(label).astype(int))
        top_idx = np.argsort(scores)[-k:][::-1]
        rows.extend(
            {
                "label": label,
                "term": feature_names[index],
                "chi2_score": float(scores[index]),
            }
            for index in top_idx
        )

    return pd.DataFrame(rows)


prompt_chi2 = top_chi2_terms(train_prepared["prompt_text"], train_prepared["target"], k=12, min_df=25)
response_a_chi2 = top_chi2_terms(train_prepared["response_a_text"], train_prepared["target"], k=12, min_df=25)
response_b_chi2 = top_chi2_terms(train_prepared["response_b_text"], train_prepared["target"], k=12, min_df=25)

print("Prompt terms with strongest chi-square signal")
display(prompt_chi2)
print("\nResponse A terms with strongest chi-square signal")
display(response_a_chi2)
print("\nResponse B terms with strongest chi-square signal")
display(response_b_chi2)

Prompt terms with strongest chi-square signal


,label,term,chi2_score
0,model_a,single dot,14.789221
1,model_a,write single,13.471108
2,model_a,dot,12.163654
3,model_a,my prompt,8.169508
4,model_a,dot and,7.744271
5,model_a,and wait,7.194665
6,model_a,sisters,6.417670
7,model_a,how many,5.803924
8,model_a,me how,5.255025
9,model_a,single,5.207268



Response A terms with strongest chi-square signal


,label,term,chi2_score
0,model_a,sorry but,33.111402
1,model_a,apologize,33.014172
2,model_a,sorry,31.262838
3,model_a,apologize but,27.405269
4,model_a,not feel,26.030320
5,model_a,feel comfortable,23.873524
6,model_a,that request,18.805771
7,model_a,but do,17.084859
8,model_a,with that,16.892500
9,model_a,assist with,16.312052



Response B terms with strongest chi-square signal


,label,term,chi2_score
0,model_a,not feel,26.265138
1,model_a,feel comfortable,24.779068
2,model_a,apologize but,16.227764
3,model_a,apologize,14.747697
4,model_a,comfortable,13.765108
5,model_a,but do,11.627006
6,model_a,comfortable generating,11.313315
7,model_a,do not,10.865082
8,model_a,perhaps we,8.487157
9,model_a,apologize upon,7.562333


In [5]:
# ─── Architecture overview ────────────────────────────────────────────────────
#
#  Three text blocks (word n-grams, shared vocabulary):
#    1. prompt_vec        – independent prompt vocabulary
#    2. response_vec      – shared response vocabulary trained on [A ∪ B]
#       response_a_block  = tfidf(resp_A)
#       response_b_block  = tfidf(resp_B)
#       diff_pos          = max(0,  resp_A - resp_B)  → words A dominates
#       diff_neg          = max(0, -[resp_A - resp_B]) → words B dominates
#
#  Chi-square selection across all five non-negative text blocks combined.
#  LogisticRegression (SAGA)
# ──────────────────────────────────────────────────────────────────────────────


def build_feature_artifacts(train_frame: pd.DataFrame, y: pd.Series, *, select_k: int = 55000) -> dict[str, object]:
    prompt_vectorizer = TfidfVectorizer(
        min_df=5,
        max_df=0.98,
        ngram_range=(1, 2),
        sublinear_tf=True,
        strip_accents="unicode",
        max_features=25000,
    )
    response_vectorizer = TfidfVectorizer(
        min_df=5,
        max_df=0.98,
        ngram_range=(1, 2),
        sublinear_tf=True,
        strip_accents="unicode",
        max_features=40000,
    )
    meta_preprocessor = ColumnTransformer(
        [
            ("num", MaxAbsScaler(), NUMERIC_FEATURES),
            ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
        ],
        sparse_threshold=1.0,
    )

    prompt_train = prompt_vectorizer.fit_transform(train_frame["prompt_text"])

    # Shared vocabulary: fit on A and B together so they live in the same space
    response_vectorizer.fit(
        pd.concat([train_frame["response_a_text"], train_frame["response_b_text"]], axis=0)
    )
    response_a_train = response_vectorizer.transform(train_frame["response_a_text"])
    response_b_train = response_vectorizer.transform(train_frame["response_b_text"])

    # Pairwise difference: encodes WHICH response has more of each term
    response_diff = response_a_train - response_b_train
    diff_pos = response_diff.maximum(0)          # terms where A > B
    diff_neg = (-response_diff).maximum(0)       # terms where B > A

    text_train = sparse.hstack(
        [prompt_train, response_a_train, response_b_train, diff_pos, diff_neg],
        format="csr",
    )
    selector = SelectKBest(chi2, k=min(select_k, text_train.shape[1] - 1))
    text_train = selector.fit_transform(text_train, y)
    meta_train = meta_preprocessor.fit_transform(train_frame)

    return {
        "prompt_vectorizer": prompt_vectorizer,
        "response_vectorizer": response_vectorizer,
        "selector": selector,
        "meta_preprocessor": meta_preprocessor,
        "train_matrix": sparse.hstack([text_train, meta_train], format="csr"),
    }


def transform_with_artifacts(frame: pd.DataFrame, artifacts: dict[str, object]) -> sparse.csr_matrix:
    prompt_block = artifacts["prompt_vectorizer"].transform(frame["prompt_text"])
    response_a_block = artifacts["response_vectorizer"].transform(frame["response_a_text"])
    response_b_block = artifacts["response_vectorizer"].transform(frame["response_b_text"])
    response_diff = response_a_block - response_b_block
    diff_pos = response_diff.maximum(0)
    diff_neg = (-response_diff).maximum(0)
    text_block = sparse.hstack(
        [prompt_block, response_a_block, response_b_block, diff_pos, diff_neg],
        format="csr",
    )
    selected_text = artifacts["selector"].transform(text_block)
    meta_block = artifacts["meta_preprocessor"].transform(frame)
    return sparse.hstack([selected_text, meta_block], format="csr")


def build_baseline_matrix(
    train_frame: pd.DataFrame, valid_frame: pd.DataFrame
) -> tuple[sparse.csr_matrix, sparse.csr_matrix, TfidfVectorizer]:
    vectorizer = TfidfVectorizer(
        min_df=5,
        max_df=0.96,
        ngram_range=(1, 2),
        sublinear_tf=True,
        strip_accents="unicode",
        max_features=65000,
    )
    train_text = (
        train_frame["prompt_text"]
        + " "
        + train_frame["response_a_text"]
        + " "
        + train_frame["response_b_text"]
    )
    valid_text = (
        valid_frame["prompt_text"]
        + " "
        + valid_frame["response_a_text"]
        + " "
        + valid_frame["response_b_text"]
    )
    return vectorizer.fit_transform(train_text), vectorizer.transform(valid_text), vectorizer


def fit_classifier(X_train: sparse.csr_matrix, y_train: pd.Series) -> LogisticRegression:
    """
    LogisticRegression with SAGA solver produces well-calibrated multinomial
    probabilities on sparse TF-IDF matrices, which is critical for log_loss.
    """
    model = LogisticRegression(
        solver="saga",
        C=0.8,
        max_iter=300,
        tol=1e-3,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    model.fit(X_train, y_train)
    return model


def evaluate_holdout(train_frame: pd.DataFrame, *, sample_size: int = 22000) -> pd.DataFrame:
    sample = train_frame.sample(
        min(sample_size, len(train_frame)), random_state=RANDOM_STATE
    ).reset_index(drop=True)
    fit_frame, valid_frame = train_test_split(
        sample,
        test_size=0.2,
        stratify=sample["target"],
        random_state=RANDOM_STATE,
    )

    # Baseline: single concatenated TF-IDF
    baseline_train, baseline_valid, _ = build_baseline_matrix(fit_frame, valid_frame)
    baseline_model = fit_classifier(baseline_train, fit_frame["target"])
    baseline_proba = baseline_model.predict_proba(baseline_valid)

    # Enhanced: separate vocab blocks + pairwise diff + chi2 + meta
    artifacts = build_feature_artifacts(fit_frame, fit_frame["target"], select_k=50000)
    enhanced_model = fit_classifier(artifacts["train_matrix"], fit_frame["target"])
    enhanced_valid = transform_with_artifacts(valid_frame, artifacts)
    enhanced_proba = enhanced_model.predict_proba(enhanced_valid)

    results = pd.DataFrame(
        [
            {
                "model": "baseline_single_tfidf_logreg",
                "log_loss": log_loss(valid_frame["target"], baseline_proba, labels=baseline_model.classes_),
                "accuracy": accuracy_score(valid_frame["target"], baseline_model.predict(baseline_valid)),
                "features": baseline_train.shape[1],
            },
            {
                "model": "multi_vocab_diff_chi2_logreg",
                "log_loss": log_loss(valid_frame["target"], enhanced_proba, labels=enhanced_model.classes_),
                "accuracy": accuracy_score(valid_frame["target"], enhanced_model.predict(enhanced_valid)),
                "features": artifacts["train_matrix"].shape[1],
            },
        ]
    ).sort_values("log_loss").reset_index(drop=True)
    return results


holdout_results = evaluate_holdout(train_prepared, sample_size=22000)
print("Holdout comparison (sorted by log_loss ↑ = better)")
display(holdout_results.round(5))


Holdout comparison (sorted by log_loss ↑ = better)


,model,log_loss,accuracy,features
0,multi_vocab_diff_chi2_logreg,1.04275,0.48682,50146
1,baseline_single_tfidf_logreg,1.09616,0.38523,65000


In [6]:
cv_sample = train_prepared.sample(min(32000, len(train_prepared)), random_state=RANDOM_STATE).reset_index(drop=True)
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
cv_rows: list[dict[str, float]] = []

for fold, (fit_idx, valid_idx) in enumerate(cv.split(cv_sample, cv_sample["target"]), start=1):
    fit_frame = cv_sample.iloc[fit_idx].reset_index(drop=True)
    valid_frame = cv_sample.iloc[valid_idx].reset_index(drop=True)

    artifacts = build_feature_artifacts(fit_frame, fit_frame["target"], select_k=50000)
    model = fit_classifier(artifacts["train_matrix"], fit_frame["target"])
    valid_matrix = transform_with_artifacts(valid_frame, artifacts)
    valid_proba = model.predict_proba(valid_matrix)

    cv_rows.append(
        {
            "fold": fold,
            "log_loss": log_loss(valid_frame["target"], valid_proba, labels=model.classes_),
            "accuracy": accuracy_score(valid_frame["target"], model.predict(valid_matrix)),
        }
    )

cv_results = pd.DataFrame(cv_rows)
print("3-fold CV for the stronger TF-IDF + chi-square pipeline")
display(cv_results.round(5))
print("\nMean CV metrics")
display(cv_results.mean(numeric_only=True).to_frame("mean").T.round(5))

3-fold CV for the stronger TF-IDF + chi-square pipeline


,fold,log_loss,accuracy
0,1,1.03502,0.48064
1,2,1.04688,0.47127
2,3,1.04350,0.47628



Mean CV metrics


,fold,log_loss,accuracy
mean,2.0,1.0418,0.47606


In [7]:
final_artifacts = build_feature_artifacts(train_prepared, train_prepared["target"], select_k=55000)
final_model = fit_classifier(final_artifacts["train_matrix"], train_prepared["target"])
test_matrix = transform_with_artifacts(test_prepared, final_artifacts)
test_proba = final_model.predict_proba(test_matrix)

submission = pd.DataFrame(test_proba, columns=final_model.classes_)
submission = submission.reindex(columns=TARGET_NAMES, fill_value=0.0)
submission.insert(0, "id", test_df["id"])
submission.to_csv("submission.csv", index=False)

joblib.dump(
    {
        "model": final_model,
        "artifacts": final_artifacts,
        "target_names": TARGET_NAMES.tolist(),
        "numeric_features": NUMERIC_FEATURES,
        "categorical_features": CATEGORICAL_FEATURES,
    },
    "tfidf_chi2_model.joblib",
)

submission.head()

,id,model_a,model_b,tie
0,136060,0.103668,0.365870,0.530462
1,211333,0.596819,0.188270,0.214910
2,1233961,0.224977,0.572825,0.202199
